In [1]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
from flax import nnx
import jax.tree_util as jtu
import sys
sys.path.insert(0, '.')


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# H2分子几何构型
bond_length = 1.4  # 平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

print("="*60)
print("H2分子系统设置")
print("="*60)
print(f"键长: {bond_length} 埃")

# 创建分子对象
mol = gto.M(atom=geometry, basis='STO-3G')

# Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"\nHartree-Fock能量: {E_hf:.8f} Ha")

# FCI计算（参考值）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print(f"\nFCI能级（参考值）:")
print("-"*50)
for i, e in enumerate(E_fcis):
    exc_energy = (e - E_fcis[0]) * 27.2114
    if i == 0:
        print(f"  E{i} (基态)     = {e:.8f} Ha")
    else:
        print(f"  E{i} (第{i}激发态) = {e:.8f} Ha  激发能: {exc_energy:.4f} eV")

# 创建NetKet哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

H2分子系统设置
键长: 1.4 埃

Hartree-Fock能量: -0.94148065 Ha

FCI能级（参考值）:
--------------------------------------------------
  E0 (基态)     = -1.01546825 Ha
  E1 (第1激发态) = -0.87542794 Ha  激发能: 3.8107 eV
  E2 (第2激发态) = -0.42938376 Ha  激发能: 15.9482 eV
  E3 (第3激发态) = -0.26922131 Ha  激发能: 20.3064 eV


In [3]:
# 创建Hilbert空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

print("="*60)
print("Hilbert空间信息")
print("="*60)
print(f"空间轨道数: 2")
print(f"自旋轨道数: 4")
print(f"电子数: 2 (α=1, β=1)")
print(f"Hilbert空间维度: {hi.n_states}")
print(f"\n所有可能的电子组态:")
print(hi.all_states())

# 创建采样器
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

Hilbert空间信息
空间轨道数: 2
自旋轨道数: 4
电子数: 2 (α=1, β=1)
Hilbert空间维度: 4

所有可能的电子组态:
[[0 1 0 1]
 [0 1 1 0]
 [1 0 0 1]
 [1 0 1 0]]


In [5]:
import jax.tree_util as jtu

# 4.1 单态 FFNN 模型（复数值输出，匹配论文架构）
class FFNN(nnx.Module):
    def __init__(self, n_orbitals: int, hidden_dim: int = 8, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_orbitals = n_orbitals
        self.hidden_dim = hidden_dim
        # 三层全连接，复数值参数
        self.linear1 = nnx.Linear(n_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output_layer = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array) -> jax.Array:
        h = self.linear1(x.astype(complex))  # 输入转为复数值
        h = nnx.tanh(h)
        h = self.linear2(h)
        h = nnx.tanh(h)
        out = self.output_layer(h)
        return jnp.squeeze(out, axis=-1)  # 输出标量复数

# 4.2 初始化 4 个单态 Ansatz（K=4，对应 FCI 的 4 个态）
rng_seeds = [30, 31]  # 论文指定随机种子
n_spin_orbitals = hi.size  # 4 个自旋轨道
hidden_dim = hi.n_orbitals * 3  # 隐藏层维度=2×3=6（论文配置）

ffnn_list = [
    FFNN(n_orbitals=n_spin_orbitals, hidden_dim=hidden_dim, rngs=nnx.Rngs(seed))
    for seed in rng_seeds
]

# 验证单态输出（复数值）
test_state = hi.all_states()[0]
print(f"\n单态 Ansatz 测试输出: {ffnn_list[0](test_state)}")  # 预期类似 0.508+0.739j



单态 Ansatz 测试输出: (0.5081542321554642+0.7394651248675597j)
